# S6.6 — Answer Generation Node

Interactive verification of the citation-backed answer generation node.
This node takes relevant sources from the agent state, invokes the LLM,
and enforces proper inline [N] citations with arXiv links.

In [1]:
import sys
sys.path.insert(0, "../..")

from src.services.agents.nodes.generate_answer_node import (
    GENERATION_PROMPT,
    NO_SOURCES_MESSAGE,
    ainvoke_generate_answer_step,
    build_generation_prompt,
    source_items_to_references,
)
from src.services.agents.models import SourceItem
from src.services.agents.state import AgentState, create_initial_state
from src.services.agents.context import AgentContext
from src.services.rag.models import SourceReference

print("\u2713 All imports successful")

✓ All imports successful


## FR-1: Prompt Construction

In [2]:
sources = [
    SourceItem(
        arxiv_id="1706.03762", title="Attention Is All You Need",
        authors=["Vaswani", "Shazeer", "Parmar"],
        url="https://arxiv.org/abs/1706.03762",
        chunk_text="The Transformer architecture replaces recurrence with self-attention.",
        relevance_score=0.95,
    ),
    SourceItem(
        arxiv_id="1810.04805", title="BERT: Pre-training of Deep Bidirectional Transformers",
        authors=["Devlin", "Chang", "Lee", "Toutanova"],
        url="https://arxiv.org/abs/1810.04805",
        chunk_text="BERT uses masked language modeling for pre-training.",
        relevance_score=0.88,
    ),
]

prompt = build_generation_prompt("What are transformers?", sources)
print(prompt)

assert "[1]" in prompt
assert "[2]" in prompt
assert "Attention Is All You Need" in prompt
assert "BERT" in prompt
print("\n\u2713 Prompt includes numbered sources and citation instructions")

You are a research assistant that answers questions using ONLY the provided academic paper excerpts.
You MUST cite sources inline using [N] notation (e.g., [1], [2]) corresponding to the numbered sources below.
Every factual claim MUST have at least one citation. Do NOT fabricate information or cite sources not provided.

If the sources do not contain enough information to answer the question, say so honestly.

## Sources

[1] Attention Is All You Need (arxiv: 1706.03762)
The Transformer architecture replaces recurrence with self-attention.

[2] BERT: Pre-training of Deep Bidirectional Transformers (arxiv: 1810.04805)
BERT uses masked language modeling for pre-training.

## Question

What are transformers?

## Instructions

1. Answer the question using ONLY information from the sources above.
2. Use inline citations [1], [2], etc. for every claim.
3. Be concise but thorough.
4. Do NOT add a "Sources" section at the end — it will be appended automatically.

✓ Prompt includes numbered so

## FR-3: SourceItem to SourceReference Conversion

In [3]:
refs = source_items_to_references(sources)

assert len(refs) == 2
assert isinstance(refs[0], SourceReference)
assert refs[0].index == 1
assert refs[0].arxiv_id == "1706.03762"
assert refs[1].index == 2
assert refs[1].arxiv_url == "https://arxiv.org/abs/1810.04805"

for ref in refs:
    print(f"  [{ref.index}] {ref.title} — {ref.arxiv_url}")

print("\n\u2713 SourceItem -> SourceReference conversion works correctly")

  [1] Attention Is All You Need — https://arxiv.org/abs/1706.03762
  [2] BERT: Pre-training of Deep Bidirectional Transformers — https://arxiv.org/abs/1810.04805

✓ SourceItem -> SourceReference conversion works correctly


## FR-5: No-Sources Fallback

In [4]:
from unittest.mock import MagicMock

mock_provider = MagicMock()
context = AgentContext(llm_provider=mock_provider)

state = AgentState(
    messages=[],
    original_query="What is dark matter?",
    relevant_sources=[],
    metadata={},
)

result = await ainvoke_generate_answer_step(state, context)

ai_msg = result["messages"][0]
print(f"Fallback message: {ai_msg.content}")
assert NO_SOURCES_MESSAGE in ai_msg.content
mock_provider.get_langchain_model.assert_not_called()

print("\n\u2713 No-sources fallback works and skips LLM call")

Fallback message: I don't have papers on that topic in my knowledge base. Please try rephrasing your question or searching for a different topic.

✓ No-sources fallback works and skips LLM call


## FR-2 + FR-4: Full Generation Flow (Mocked LLM)

In [5]:
from unittest.mock import AsyncMock
from langchain_core.messages import AIMessage, HumanMessage

# Mock LLM that returns a cited answer
mock_llm = AsyncMock()
mock_llm.ainvoke = AsyncMock(
    return_value=AIMessage(
        content="Transformers [1] use self-attention instead of recurrence. "
        "BERT [2] applies this for pre-training via masked LM."
    )
)

mock_provider = MagicMock()
mock_provider.get_langchain_model.return_value = mock_llm

context = AgentContext(llm_provider=mock_provider, model_name="test")
state = AgentState(
    messages=[HumanMessage(content="What are transformers?")],
    original_query="What are transformers?",
    relevant_sources=sources,
    metadata={},
)

result = await ainvoke_generate_answer_step(state, context)

ai_msg = result["messages"][0]
print("Generated answer:")
print(ai_msg.content)
print()

meta = result["metadata"]["citation_validation"]
print(f"Citations valid: {meta['is_valid']}")
print(f"Valid citations: {meta['valid_citations']}")
print(f"Coverage: {meta['citation_coverage']:.0%}")

assert "**Sources:**" in ai_msg.content
assert meta["is_valid"] is True

print("\n\u2713 Full generation flow with citation enforcement works")

Generated answer:
Transformers [1] use self-attention instead of recurrence. BERT [2] applies this for pre-training via masked LM.

**Sources:**
1. [Attention Is All You Need](https://arxiv.org/abs/1706.03762) — Vaswani, Shazeer, Parmar, 2017
2. [BERT: Pre-training of Deep Bidirectional Transformers](https://arxiv.org/abs/1810.04805) — Devlin et al., 2018

Citations valid: True
Valid citations: [1, 2]
Coverage: 100%

✓ Full generation flow with citation enforcement works


## Summary

All S6.6 functional requirements verified:
- FR-1: Prompt construction with numbered sources
- FR-2: LLM answer generation
- FR-3: Citation post-processing (SourceItem -> SourceReference + enforce_citations)
- FR-4: State update with AIMessage
- FR-5: No-sources fallback